# SerpentGuard AI — Extended Sources Data Pull

Pulls SnakeCLEF 2023, Open Images V7, and Kaggle India snake datasets.
Runs AFTER 01_build_raw_dataset. Internet must be enabled.

In [ ]:
import numpy as np, pandas as pd, requests, json, os, shutil
from pathlib import Path
import kagglehub
from PIL import Image
import imagehash

KAGGLE_USERNAME = "puneetbajajiitm"
SERPENTES_TAXON_ID = 85553
# Species taxon IDs from classes.csv (our 208 classes)
# Load dynamically:
splits_path = Path(kagglehub.dataset_download("puneetbajajiitm/serpentguard-m2-splits"))
classes_df = pd.read_csv(splits_path / "classes.csv")
OUR_TAXON_IDS = set(classes_df["taxon_id"].dropna().astype(int).tolist())
OUR_SPECIES = set(classes_df["species"].tolist())
print(f"Our 208 species loaded. Taxon IDs: {len(OUR_TAXON_IDS)}")

## Source 1: SnakeCLEF 2023 @ 500px

In [ ]:
# Attach dataset: munhamrehman/snakeclef2023-500px
# This dataset has ~17 GB of snake images from 1784 species worldwide
# We filter to: (1) our 208 species by taxon_id, OR (2) country == India

SNAKECLEF_PATH = Path("/kaggle/input/snakeclef2023-500px")
manifest_rows = []

if SNAKECLEF_PATH.exists():
    # Look for metadata CSV
    meta_files = list(SNAKECLEF_PATH.glob("**/*.csv"))
    print(f"Found CSV files: {[f.name for f in meta_files]}")
    
    for mf in meta_files[:1]:  # use first CSV
        sclef_df = pd.read_csv(mf)
        print(f"SnakeCLEF metadata: {sclef_df.shape}, columns: {sclef_df.columns.tolist()}")
        
        # Filter to India OR our species
        mask = pd.Series([False]*len(sclef_df))
        if "country" in sclef_df.columns:
            mask |= (sclef_df["country"].str.lower() == "india")
        if "taxon_id" in sclef_df.columns:
            mask |= sclef_df["taxon_id"].isin(OUR_TAXON_IDS)
        if "binomial" in sclef_df.columns or "species" in sclef_df.columns:
            sp_col = "binomial" if "binomial" in sclef_df.columns else "species"
            mask |= sclef_df[sp_col].isin(OUR_SPECIES)
        
        filtered = sclef_df[mask].copy()
        filtered["source"] = "snakeclef2023"
        manifest_rows.append(filtered)
        print(f"SnakeCLEF India/our-species subset: {len(filtered)} rows")
else:
    print("SnakeCLEF dataset not attached. Attach munhamrehman/snakeclef2023-500px to use this source.")

snakeclef_df = pd.concat(manifest_rows) if manifest_rows else pd.DataFrame()
print(f"SnakeCLEF rows collected: {len(snakeclef_df)}")

## Source 2: Kaggle India-specific datasets

In [ ]:
# Datasets to try (attach as input):
# - adityasharma01/snake-dataset-india  (65 MB, venomous/non-venomous)
# - aishanikaggle/indian-snake           (169 MB, CC BY-NC-SA)

india_rows = []
INDIA_PATHS = [
    Path("/kaggle/input/snake-dataset-india"),
    Path("/kaggle/input/indian-snake"),
]

# Manual species name mapping (folder label -> scientific name)
INDIA_LABEL_MAP = {
    "cobra": "Naja naja",
    "indian cobra": "Naja naja",
    "krait": "Bungarus caeruleus",
    "common krait": "Bungarus caeruleus",
    "russells viper": "Daboia russelii",
    "russell viper": "Daboia russelii",
    "saw scaled viper": "Echis carinatus",
    "rat snake": "Ptyas mucosa",
    "checkered keelback": "Fowlea piscator",
    "python": "Python molurus",
    "sand boa": "Eryx conicus",
    "vine snake": "Ahaetulla oxyrhynca",
    "wolf snake": "Lycodon aulicus",
    "cat snake": "Boiga trigonata",
}

for ip in INDIA_PATHS:
    if ip.exists():
        for species_dir in sorted(ip.iterdir()):
            if species_dir.is_dir():
                folder_name = species_dir.name.lower().strip()
                sci_name = INDIA_LABEL_MAP.get(folder_name)
                if sci_name and sci_name in OUR_SPECIES:
                    imgs = list(species_dir.glob("**/*.jpg")) + list(species_dir.glob("**/*.jpeg")) + list(species_dir.glob("**/*.png"))
                    for img_path in imgs:
                        india_rows.append({
                            "file": str(img_path),
                            "source": ip.name,
                            "species": sci_name,
                            "license": "cc_by_nc_sa" if "indian-snake" in ip.name else "unknown",
                        })
                    print(f"  {ip.name}/{species_dir.name} -> {sci_name}: {len(imgs)} images")
    else:
        print(f"Not attached: {ip}")

india_df = pd.DataFrame(india_rows) if india_rows else pd.DataFrame()
print(f"India datasets rows: {len(india_df)}")

## Source 3: Open Images V7 — Snake detection boxes

In [ ]:
# Open Images V7 snake detection download
# Uses fiftyone library (available in Kaggle)
# Snake class ID in Open Images: /m/078jl

OID_OUTPUT = Path("/kaggle/working/open_images_snakes")
OID_OUTPUT.mkdir(exist_ok=True)

try:
    import fiftyone as fo
    import fiftyone.zoo as foz
    
    print("Downloading Open Images V7 snake detection data (validation + test splits first)...")
    for split in ["validation", "test"]:
        try:
            ds = foz.load_zoo_dataset(
                "open-images-v7",
                split=split,
                label_types=["detections"],
                classes=["Snake"],
                dataset_name=f"oi_snakes_{split}",
                max_samples=500,
            )
            df_oid = ds.to_dataframes()["detections"]
            df_oid["split"] = split
            df_oid.to_csv(OID_OUTPUT / f"oid_{split}.csv", index=False)
            print(f"  {split}: {len(ds)} images with snake boxes")
        except Exception as e:
            print(f"  {split} failed: {e}")
    
    # Save YOLO-format labels
    yolo_dir = OID_OUTPUT / "yolo"
    yolo_dir.mkdir(exist_ok=True)
    # (YOLO export code would go here)
    
except ImportError:
    print("fiftyone not available. Install: pip install fiftyone")
    print("Alternative: download Open Images snake subset from:")
    print("  https://storage.googleapis.com/openimages/web/download_v7.html")
    print("  Class: Snake (/m/078jl)")
    
    # Fallback: download via OIDv4 toolkit
    oid_manifest_url = "https://storage.googleapis.com/openimages/2018_04/validation/validation-annotations-bbox.csv"
    print(f"Downloading OID bbox manifest from: {oid_manifest_url}")

## Source 4: GBIF India snake occurrences (with images)

In [ ]:
# GBIF bulk download for India snake occurrences with images
# Requires GBIF account credentials

GBIF_USER = os.environ.get("GBIF_USER", "")
GBIF_PWD  = os.environ.get("GBIF_PWD", "")

if GBIF_USER and GBIF_PWD:
    import base64
    creds = base64.b64encode(f"{GBIF_USER}:{GBIF_PWD}".encode()).decode()
    
    # Request GBIF download (async - takes minutes to prepare)
    predicate = {
        "type": "and",
        "predicates": [
            {"type": "equals", "key": "ORDER", "value": "Squamata"},
            {"type": "equals", "key": "CLASS", "value": "Reptilia"},
            {"type": "equals", "key": "COUNTRY", "value": "IN"},
            {"type": "equals", "key": "HAS_COORDINATE", "value": "true"},
            {"type": "equals", "key": "HAS_GEOSPATIAL_ISSUE", "value": "false"},
            {"type": "equals", "key": "MEDIA_TYPE", "value": "StillImage"},
        ]
    }
    
    resp = requests.post(
        "https://api.gbif.org/v1/occurrence/download/request",
        json={"predicate": predicate},
        headers={"Authorization": f"Basic {creds}", "Content-Type": "application/json"}
    )
    
    if resp.status_code == 201:
        download_key = resp.text.strip()
        print(f"GBIF download requested. Key: {download_key}")
        print("Poll: https://api.gbif.org/v1/occurrence/download/{download_key}")
        print("Download will be ready in 5-30 minutes via email link.")
        
        # Save key for later
        with open("/kaggle/working/gbif_download_key.txt", "w") as f:
            f.write(download_key)
    else:
        print(f"GBIF request failed: {resp.status_code} {resp.text[:200]}")
else:
    print("GBIF credentials not set. Set env vars GBIF_USER and GBIF_PWD.")
    print("Register at: https://www.gbif.org/user/profile")
    print("GBIF India snake occurrence data also available at:")
    print("  https://www.gbif.org/occurrence/search?country=IN&taxon_key=11418114&media_type=StillImage")

## Source 5: Roboflow Universe snake detection

In [ ]:
# Roboflow Universe snake detection datasets
# Known relevant projects:
ROBOFLOW_PROJECTS = [
    ("snakes-zjtwh", "snakes-zjtwh"),           # snake detection
    ("snake-detection-dataset", "ygzaw"),        # general snake detection
    ("serpent-detection", "serpent-detection"),  # cobra/viper focus
]

ROBOFLOW_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
rf_rows = []

if ROBOFLOW_KEY:
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=ROBOFLOW_KEY)
        
        output_det = Path("/kaggle/working/roboflow_detection")
        output_det.mkdir(exist_ok=True)
        
        for workspace, project_name in ROBOFLOW_PROJECTS:
            try:
                project = rf.workspace(workspace).project(project_name)
                version = project.versions()[0]
                dataset = version.download("yolov8", location=str(output_det / project_name))
                print(f"Downloaded: {workspace}/{project_name}")
                
                # Count images and boxes
                imgs = list((output_det / project_name).glob("**/*.jpg"))
                labels = list((output_det / project_name).glob("**/*.txt"))
                rf_rows.append({"project": project_name, "images": len(imgs), "labels": len(labels)})
            except Exception as e:
                print(f"  Failed {workspace}/{project_name}: {e}")
    except ImportError:
        print("roboflow not installed. Install: pip install roboflow")
else:
    print("ROBOFLOW_API_KEY not set. Visit: https://roboflow.com/universe")
    print("Search 'snake detection' to find relevant datasets.")
    
    # Fallback: list known public datasets
    public_datasets = [
        "https://universe.roboflow.com/lj-gkmpv/snakes-zjtwh",
        "https://universe.roboflow.com/summerflight/common-snake-species-dataset-for-object-detection",
    ]
    print("Known public datasets:")
    for url in public_datasets:
        print(f"  {url}")

rf_df = pd.DataFrame(rf_rows) if rf_rows else pd.DataFrame()
print(f"Roboflow detection datasets: {len(rf_df)}")

## Summary: Extended sources manifest

In [ ]:
# Compile all new data into extended_manifest.csv
all_dfs = []
if len(snakeclef_df) > 0: all_dfs.append(snakeclef_df)
if len(india_df) > 0: all_dfs.append(india_df)

if all_dfs:
    extended_df = pd.concat(all_dfs, ignore_index=True)
    extended_df.to_csv("/kaggle/working/extended_manifest.csv", index=False)
    print(f"Extended manifest saved: {len(extended_df)} rows")
    print(extended_df["source"].value_counts().to_string())
else:
    print("No new sources collected in this run.")
    print("To collect data: attach the datasets listed in this notebook as inputs.")
    
# Detection manifest
if OID_OUTPUT.exists():
    oid_csvs = list(OID_OUTPUT.glob("*.csv"))
    if oid_csvs:
        det_df = pd.concat([pd.read_csv(f) for f in oid_csvs], ignore_index=True)
        det_df.to_csv("/kaggle/working/detection_manifest.csv", index=False)
        print(f"Detection manifest: {len(det_df)} rows")

print("\nNext steps:")
print("1. Attach datasets to this notebook and re-run")
print("2. Run 09_combine_and_reclean.ipynb to merge with iNaturalist base")
print("3. Run 10_eda_full.ipynb to see cross-source EDA")